In [ ]:
#do testowania

data_id_enriched_market = input.run_Import("/workspaces/SolvMate/input/02.01_SAS_Input_MarketR.xls", ['Basic input','MarketR'])

import external_assets as ex

reader = ex.SupabaseDataFrameReader()
ext_assets = reader.read_from_supabase("external_market_risk_input")
currencies = reader.read_from_supabase("exchange_rates")


reporting_date = '2020-12-31'
entity_id = 'SCE001'
filtered_ex_ass = ex_ass[(ex_ass['REPORTING_DT'] == reporting_date) & (ex_ass['ENTITY_ID'] == entity_id)]

results = []
for data_id in market_external_assets_data_id:
    results.append(filtered_ex_ass[filtered_ex_ass['DATA_ID'] == data_id][['DATA_ID', 'VALUE']])

results_df = pd.concat(results, ignore_index=True)

waluty = reader.read_from_supabase("exchange_rates")

spot_rate = waluty.loc[(waluty['REPORTING_DT'] == reporting_date) & (waluty['TO_CURRENCY'] == "PLN"), 'SPOT_RATE'].iloc[0]
results_df = results_df.copy()
results_df['VALUE'] = results_df['VALUE'] * spot_rate

data_id_enriched_market = data_id_enriched_market.copy()
data_id_enriched_market.set_index('DATA_ID', inplace=True)
results_df.set_index('DATA_ID', inplace=True)

# Update VALUE for matching DATA_IDs
data_id_enriched_market.loc[results_df.index, 'VALUE'] = results_df['VALUE']

data_id_enriched_market.reset_index(inplace=True)
